# BIP-39 and BIP-32: Key Derivation with bip_utils

Companion to [BIP-39 and BIP-32: Understanding Key Derivation](https://weisser-zwerg.dev/posts/bip39-bip32-key-derivation/).

This concise version uses `bip_utils==2.12.3` to generate the recovery phrase and seed, then derive Bitcoin and Ethereum private/public keys.
Its inputs and outputs match the [by-foot notebook](bip39-bip32-key-derivation-by-foot.ipynb).

In [1]:
%load_ext watermark
%watermark -a 'cs224' -u -d -v -p bip_utils,notebook,nbclassic,watermark

Author: cs224

Last updated: 2026-11-25

Python implementation: CPython
Python version       : 3.12.9
IPython version      : 9.19.1

bip_utils: 2.12.3
notebook : 7.6.5
nbclassic: 1.3.3
watermark: 2.6.0



In [2]:
from IPython.display import HTML, display

# Notebook 6 and nbclassic
# https://github.com/jupyter/nbclassic
display(HTML("<style>.container { width:70% !important; }</style>"))

# Notebook 7
display(HTML("<style>:root { --jp-notebook-max-width: 70% !important; }</style>"))

## Public Example Recovery Phrases

We use the public all-zero-entropy BIP-39 examples: `abandon` repeated 11 times followed by `about`, or 23 times followed by `art`.
These predictable phrases are for learning and must never hold funds.

Use `ENTROPY_BITS = 128` or `256`, with an empty `BIP39_PASSPHRASE` by default.
Change the selected inputs and rerun the notebook from the beginning.

In [3]:
MNEMONIC_12 = "abandon " * 11 + "about"
MNEMONIC_24 = "abandon " * 23 + "art"

ENTROPY_BITS = 128  # Use 256 for the 24-word example.
BIP39_PASSPHRASE = ""

print("12-word public example:", MNEMONIC_12)
print("24-word public example:", MNEMONIC_24)

12-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about
24-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon art


## From Entropy to Seed

The library's [BIP-39 functions](https://github.com/ebellocchia/bip_utils/blob/v2.12.3/readme/bip39.md) generate the checksum and words, validate and decode the phrase, and derive the 64-byte seed.
The same entropy and passphrase reproduce the by-foot results.

In [4]:
from bip_utils import (
    Bip39MnemonicDecoder,
    Bip39MnemonicGenerator,
    Bip39SeedGenerator,
)

entropy = bytes(ENTROPY_BITS // 8)
mnemonic = str(Bip39MnemonicGenerator().FromEntropy(entropy))
assert Bip39MnemonicDecoder().Decode(mnemonic) == entropy
seed = Bip39SeedGenerator(mnemonic).Generate(BIP39_PASSPHRASE)

print("Entropy:", entropy.hex())
print("Recovery phrase:", mnemonic)
print("Seed:", seed.hex())
print("Seed length (bytes):", len(seed))

Entropy: 00000000000000000000000000000000
Recovery phrase: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about
Seed: 5eb00bbddcf069084889a8ab9155568165f5c453ccb85e70811aaed6f6da5fc19a5ac40b389cd370d086206dec8aa6c43daea6690f20ad3d8d48b2d2ce9e38e4
Seed length (bytes): 64


## Master Node

Use the library's secp256k1 BIP-32 class to obtain the master private key and chain code from the seed.

In [5]:
from bip_utils import Bip32Secp256k1

master = Bip32Secp256k1.FromSeed(seed)
print("Master private key:", master.PrivateKey().Raw().ToHex())
print("Master chain code:", master.ChainCode().ToHex())

Master private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
Master chain code: 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e


## Bitcoin and Ethereum Paths

Use `m/84'/0'/0'/0/0` for the BIP-84 Bitcoin example and `m/44'/60'/0'/0/0` for the Ethereum example.
[`DerivePath()`](https://github.com/ebellocchia/bip_utils/blob/v2.12.3/readme/bip32.md) performs the hardened and normal derivations and returns the selected child node.
The printed keys and chain codes match the by-foot implementation.

In [6]:
paths = {
    "Bitcoin": "m/84'/0'/0'/0/0",
    "Ethereum": "m/44'/60'/0'/0/0",
}

derived = {}
for name, path in paths.items():
    child = master.DerivePath(path)
    derived[name] = child
    print(f"\n{name}: {path}")
    print("Private key:", child.PrivateKey().Raw().ToHex())
    print("Compressed public key:", child.PublicKey().RawCompressed().ToHex())
    print("Chain code:", child.ChainCode().ToHex())


Bitcoin: m/84'/0'/0'/0/0
Private key: 4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3
Compressed public key: 0330d54fd0dd420a6e5f8d3624f5f3482cae350f79d5f0753bf5beef9c2d91af3c
Chain code: ab52cce8c3dc905c885ff8aa6d3374df3c27a38363acd3d623791b45017577ed

Ethereum: m/44'/60'/0'/0/0
Private key: 1ab42cc412b618bdea3a599e3c9bae199ebf030895b039e9db1e30dafb12b727
Compressed public key: 0237b0bb7a8288d38ed49a524b5dc98cff3eb5ca824c9f9dc0dfdb3d9cd600f299
Chain code: 736094f4f24b67e838a4b3d23d31d229ca03e00c9bb99ce95da6d86e8b3847b5


## Check Published Examples

Use the same reference values as the by-foot notebook:

- [BIP-39](https://github.com/trezor/python-mnemonic/blob/master/vectors.json): the two public phrases with the additional passphrase `TREZOR`.
- [BIP-32](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki#test-vectors): a master node, a mixed path, and a leading-zero case.
- [BIP-84](https://github.com/bitcoin/bips/blob/master/bip-0084.mediawiki#test-vectors): two receiving keys and one change key, using the public 12-word phrase and an empty passphrase.

These fixed checks remain independent of the selected inputs for the main example.

In [7]:
reference_vectors = [
    (
        MNEMONIC_12,
        "c55257c360c07c72029aebc1b53c05ed0362ada38ead3e3e9efa3708e5349553"
        "1f09a6987599d18264c1e1c92f2cf141630c7a3c4ab7c81b2f001698e7463b04",
    ),
    (
        MNEMONIC_24,
        "bda85446c68413707090a52022edd26a1c9462295029f2e60cd7c4f2bbd30971"
        "70af7a4d73245cafa9c3cca8d561a7c3de6f5d4a10be8ed2a5e608d68f92fcc8",
    ),
]

for reference_mnemonic, expected_seed in reference_vectors:
    actual_seed = Bip39SeedGenerator(reference_mnemonic).Generate("TREZOR")
    assert actual_seed.hex() == expected_seed
    print(f"{len(reference_mnemonic.split())}-word published seed matches")

12-word published seed matches
24-word published seed matches


In [8]:
bip32_reference_nodes = [
    (
        "000102030405060708090a0b0c0d0e0f",
        "m",
        "e8f32e723decf4051aefac8e2c93c9c5b214313817cdb01a1494b917c8436b35",
        "873dff81c02f525623fd1fe5167eac3a55a049de3d314bb42ee227ffed37d508",
    ),
    (
        "000102030405060708090a0b0c0d0e0f",
        "m/0'/1/2'/2/1000000000",
        "471b76e389e528d6de6d816857e012c5455051cad6660850e58372a6c3e6e7c8",
        "c783e67b921d2beb8f6b389cc646d7263b4145701dadd2161548a8b078e65e9e",
    ),
    (
        "3ddd5602285899a946114506157c7997e5444528f3003f6134712147db19b678",
        "m/0'/1'",
        "3a2086edd7d9df86c3487a5905a1712a9aa664bce8cc268141e07549eaa8661d",
        "a48ee6674c5264a237703fd383bccd9fad4d9378ac98ab05e6e7029b06360c0d",
    ),
]

for seed_hex, path, expected_private, expected_chain in bip32_reference_nodes:
    reference = Bip32Secp256k1.FromSeed(bytes.fromhex(seed_hex)).DerivePath(path)
    assert reference.PrivateKey().Raw().ToHex() == expected_private
    assert reference.ChainCode().ToHex() == expected_chain
    print("Published BIP-32 node matches:", path)

Published BIP-32 node matches: m
Published BIP-32 node matches: m/0'/1/2'/2/1000000000
Published BIP-32 node matches: m/0'/1'


In [9]:
reference_master = Bip32Secp256k1.FromSeed(
    Bip39SeedGenerator(MNEMONIC_12).Generate("")
)
bip84_reference_nodes = [
    (
        "m/84'/0'/0'/0/0",
        "4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3",
        "0330d54fd0dd420a6e5f8d3624f5f3482cae350f79d5f0753bf5beef9c2d91af3c",
    ),
    (
        "m/84'/0'/0'/0/1",
        "2fd0affa51529f940a358ec0c50de81267d0bf5158ca61887347676946362c5b",
        "03e775fd51f0dfb8cd865d9ff1cca2a158cf651fe997fdc9fee9c1d3b5e995ea77",
    ),
    (
        "m/84'/0'/0'/1/0",
        "3277578a56b721e4c9f071f1e24aa0f94c4ff72e7967fea03b134f605f07c8fd",
        "03025324888e429ab8e3dbaf1f7802648b9cd01e9b418485c5fa4c1b9b5700e1a6",
    ),
]

for path, expected_private, expected_public in bip84_reference_nodes:
    reference = reference_master.DerivePath(path)
    assert reference.PrivateKey().Raw().ToHex() == expected_private
    assert reference.PublicKey().RawCompressed().ToHex() == expected_public
    print("Published BIP-84 key matches:", path)

Published BIP-84 key matches: m/84'/0'/0'/0/0
Published BIP-84 key matches: m/84'/0'/0'/0/1
Published BIP-84 key matches: m/84'/0'/0'/1/0


We have reproduced the by-foot derivations using the higher-level library API.
The recovery recipe still consists of the phrase, any BIP-39 passphrase, and the actual derivation path.